In [6]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from collections import defaultdict
from collections import Counter
import os
import glob
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

In [7]:
ROOT    = Path(r'C:\Users\guigo\OneDrive\Escritorio\TFG_Biopsias\Proyecto')
H5_FOLDER = ROOT / 'h5_multilabel'
STATISTICS_FOLDER = Path(r'C:\Users\guigo\OneDrive\Escritorio\TFG_Biopsias\Proyecto\Statistics')
H5_FILES = list(Path(H5_FOLDER).glob("*.h5"))
CLASS_NAMES = [
    'normal',
    'lowgrade_dysplasia',
    'inflammation',
    'highgrade_dysplasia',
    'tumor_necrosis',
    'suspicious_for_invasion',
    'adenocarcinoma',
]
NUM_CLASSES = len(CLASS_NAMES)

In [8]:
# =========================
# ANALISIS DISTRIBUCIÓN DATASET
# =========================

# Total de patches por clase
class_patch_counts = np.zeros(NUM_CLASSES, dtype=int)

# Total WSI por clase
class_wsi_counts = np.zeros(NUM_CLASSES, dtype=int)

# Resumen por WSI
wsi_summary = []

for h5_path in H5_FILES:
    with h5py.File(h5_path, "r") as f:

        labels = f["labels"][:]

        # Patches por clase
        per_class_counts = labels.sum(axis=0).astype(int)

        # Clases aparecen al menos una vez
        present_classes = np.where(per_class_counts > 0)[0]

        # Actualizar contadores globales
        class_patch_counts += per_class_counts

        # Presencia en WSI
        for c in present_classes:
            class_wsi_counts[c] += 1

        # Resumen por WSI
        wsi_summary.append({
            "wsi": h5_path.name,
            "classes_present": ", ".join([CLASS_NAMES[i] for i in present_classes]),
            **{CLASS_NAMES[i]: int(per_class_counts[i]) for i in range(NUM_CLASSES)}
        })

# Tabla global por clase
global_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "total_patches": class_patch_counts,
    "num_wsis": class_wsi_counts,
})

# Calculo de porcentaje por clase
global_df["patch_percentage"] = (
    global_df["total_patches"] /
    global_df["total_patches"].sum()
) * 100

global_df = global_df.sort_values("total_patches")

print(global_df)

# Guardar CSVs
global_df.to_csv(STATISTICS_FOLDER / "global_class_stats.csv", index=False)

wsi_df = pd.DataFrame(wsi_summary)
wsi_df.to_csv(STATISTICS_FOLDER / "wsi_summary.csv", index=False)


                     class  total_patches  num_wsis  patch_percentage
4           tumor_necrosis            622        10          0.640999
5  suspicious_for_invasion            673        13          0.693557
2             inflammation           1026        22          1.057340
3      highgrade_dysplasia           3047        35          3.140072
6           adenocarcinoma           4561        34          4.700317
0                   normal          30959       174         31.904654
1       lowgrade_dysplasia          56148       115         57.863061


In [9]:
# =========================
# ANALISIS CLASES MINORITARIAS
# =========================

minority_classes = [
    'tumor_necrosis',
    'suspicious_for_invasion',
    'inflammation'
]

rows = []

for h5_path in H5_FILES:
    with h5py.File(h5_path, "r") as f:

        labels = f["labels"][:]

        # Patches por clase
        per_class_counts = labels.sum(axis=0).astype(int)

        # Patches totales
        total_patches = labels.shape[0]

        # Clases presentes en la WSI
        present_classes = np.where(per_class_counts > 0)[0]

        # Comprobar minoritarias
        for class_name in minority_classes:

            class_idx = CLASS_NAMES.index(class_name)

            if per_class_counts[class_idx] > 0:

                rows.append({
                    "target_class": class_name,
                    "wsi": h5_path.name,
                    "total_patches": total_patches,
                    "target_class_patches":
                        per_class_counts[class_idx],
                    "classes_present":
                        ", ".join([CLASS_NAMES[i] for i in present_classes]),
                    }
                )


minority_df = pd.DataFrame(rows)

# Ordenar por clase y numero de patches
minority_df = minority_df.sort_values(
    by=[
        "target_class",
        "target_class_patches"
    ],
    ascending=[True, False]
)

# Guardar CSVs
minority_df.to_csv(STATISTICS_FOLDER / "minority_class_analysis.csv", index=False)

print(minority_df.head(20))

    target_class                wsi  total_patches  target_class_patches  \
9   inflammation  090_multilabel.h5            865                   152   
40  inflammation  185_multilabel.h5            143                   116   
7   inflammation  073_multilabel.h5            207                   112   
10  inflammation  092_multilabel.h5            350                   100   
19  inflammation  119_multilabel.h5           1625                    90   
42  inflammation  190_multilabel.h5            219                    76   
21  inflammation  121_multilabel.h5            313                    70   
43  inflammation  192_multilabel.h5             60                    60   
28  inflammation  148_multilabel.h5            500                    52   
38  inflammation  184_multilabel.h5            195                    45   
23  inflammation  139_multilabel.h5            366                    29   
0   inflammation  002_multilabel.h5            176                    26   
44  inflamma

In [10]:
# =========================
# ANALISIS MULTILABEL
# =========================

rows = []

global_counter = Counter()

global_wsi_tracker = {}

for h5_path in H5_FILES:
    with h5py.File(h5_path, "r") as f:

        labels = f["labels"][:]

        # Suma por patch
        label_counts = labels.sum(axis=1)

        # Patches multilabel
        multilabel_mask = label_counts > 1

        multilabel_indices = np.where(multilabel_mask)[0]

        # Si no hay multilabel, skip
        if len(multilabel_indices) == 0:
            continue

        # Contar combinaciones dentro de esta WSI
        counter = Counter()

        for idx in multilabel_indices:

            active_classes = np.where(labels[idx] > 0)[0]

            combo = tuple(
                CLASS_NAMES[i]
                for i in active_classes
            )

            counter[combo] += 1
            global_counter[combo] += 1
            
            if combo not in global_wsi_tracker:
                global_wsi_tracker[combo] = set()
            global_wsi_tracker[combo].add(h5_path.name)

        # Guardar resumen WSI
        rows.append({
            "wsi": h5_path.name,
            "total_patches": labels.shape[0],
            "multilabel_patches": len(multilabel_indices),
            "multilabel_percentage (%)":
                round(
                    (len(multilabel_indices) / labels.shape[0]) * 100,
                    2
                ),
            "unique_multilabel_combinations":
                len(counter),

            "combinations_found":
                " || ".join([
                    f"{' + '.join(combo)} ({count})"
                    for combo, count in counter.items()
                ])
        })


multilabel_df = pd.DataFrame(rows)

multilabel_df = multilabel_df.sort_values(
    by="multilabel_patches",
    ascending=False
)

combo_rows = []

for combo, count in global_counter.items():
    
    num_wsi = len(global_wsi_tracker.get(combo, set()))
    
    if num_wsi == 0 and count > 0:
        num_wsi = 1
    
    combo_rows.append({
        "combination": " + ".join(combo),
        "num_patches": count,
        "num_WSI": num_wsi,
        "num_classes": len(combo)
    })

combo_df = pd.DataFrame(combo_rows)

combo_df = combo_df.sort_values(
    by="num_patches",
    ascending=False
)

# Guardar CSVs
multilabel_df.to_csv(
    STATISTICS_FOLDER / "multilabel_wsi_analysis.csv",
    index=False
)

print(multilabel_df.head(20))

combo_df.to_csv(
    STATISTICS_FOLDER / "multilabel_combinations_global.csv",
    index=False
)

print(combo_df)

                  wsi  total_patches  multilabel_patches  \
7   093_multilabel.h5           1795                 353   
6   090_multilabel.h5            865                 152   
13  121_multilabel.h5            313                 132   
20  171_multilabel.h5           1816                 124   
23  175_multilabel.h5           2974                  81   
9   095_multilabel.h5           3382                  79   
3   059_multilabel.h5           2997                  68   
8   094_multilabel.h5            290                  51   
26  185_multilabel.h5            143                  25   
18  165_multilabel.h5            173                  23   
1   035_multilabel.h5            195                  23   
11  106_multilabel.h5            308                  22   
10  098_multilabel.h5            221                  18   
30  198_multilabel.h5            582                  16   
17  158_multilabel.h5            280                  12   
27  189_multilabel.h5            125    

In [11]:
# =========================
# CREACIÓN FOLDS
# =========================

wsi_data = []

for h5_path in H5_FILES:
    with h5py.File(h5_path, "r") as f:
        labels = f["labels"][:]
        
        # Matriz binaria de presencia
        
        #presence_classes = np.max(labels, axis=0) 
        
        # Patches por clase
        per_class_counts = labels.sum(axis=0).astype(int)
        
        patch_count = len(labels)
        
        presence_classes = (per_class_counts >= 10).astype(int)
        wsi_big = int(patch_count > 1000)
        presence_amp = np.append(presence_classes, wsi_big)
        
        # Guardamos los datos de presencia y desglose de patches
        wsi_entry = {
            "archivo": os.path.basename(h5_path),
            "total patches": patch_count,
            "clases_presencia": presence_amp
        }
        
        # Añadimos de forma dinámica una columna por cada clase con su número de patches
        for idx_class, name_class in enumerate(CLASS_NAMES):
            wsi_entry[name_class] = per_class_counts[idx_class]
            
        wsi_data.append(wsi_entry)

# Convertir a matrices limpias para el algoritmo de estratificación
X = np.array([d["archivo"] for d in wsi_data])
Y = np.array([d["clases_presencia"] for d in wsi_data])  

# Aplicar la estratificación multilabel WSI wise
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=52)
map_folds = {}

for fold_idx, (train_idx, val_idx) in enumerate(mskf.split(X, Y)):
    for idx in val_idx:
        map_folds[X[idx]] = fold_idx

# Dataframe final uniendo conteos de parches y asignación del Fold
df_total = pd.DataFrame(wsi_data)

print(map_folds)

# Mapeamos el fold correspondiente a cada archivo usando el diccionario generado por mskf
df_total["fold"] = df_total["archivo"].map(map_folds)

# Reordenamos las columnas básicas al principio
ordered_columns = ["archivo", "fold", "total patches"] + CLASS_NAMES
df_final = df_total[ordered_columns]
df_final_by_fold = df_final.sort_values(by="fold", ascending=True).reset_index(drop=True)

# Guardar CSV
df_final_by_fold.to_csv(STATISTICS_FOLDER / "analysis_wsi_folds.csv", index=False)

print(df_final_by_fold.head().to_string())


{np.str_('005_multilabel.h5'): 0, np.str_('007_multilabel.h5'): 0, np.str_('010_multilabel.h5'): 0, np.str_('013_multilabel.h5'): 0, np.str_('019_multilabel.h5'): 0, np.str_('025_multilabel.h5'): 0, np.str_('038_multilabel.h5'): 0, np.str_('040_multilabel.h5'): 0, np.str_('044_multilabel.h5'): 0, np.str_('045_multilabel.h5'): 0, np.str_('048_multilabel.h5'): 0, np.str_('055_multilabel.h5'): 0, np.str_('059_multilabel.h5'): 0, np.str_('064_multilabel.h5'): 0, np.str_('070_multilabel.h5'): 0, np.str_('085_multilabel.h5'): 0, np.str_('087_multilabel.h5'): 0, np.str_('090_multilabel.h5'): 0, np.str_('091_multilabel.h5'): 0, np.str_('101_multilabel.h5'): 0, np.str_('105_multilabel.h5'): 0, np.str_('109_multilabel.h5'): 0, np.str_('113_multilabel.h5'): 0, np.str_('122_multilabel.h5'): 0, np.str_('123_multilabel.h5'): 0, np.str_('124_multilabel.h5'): 0, np.str_('127_multilabel.h5'): 0, np.str_('144_multilabel.h5'): 0, np.str_('145_multilabel.h5'): 0, np.str_('148_multilabel.h5'): 0, np.str_('

In [12]:
# =========================
# VERIFICACIÓN WSI WISE
# =========================

df = pd.read_csv(STATISTICS_FOLDER / "analysis_wsi_folds.csv")

print(f"Total de filas analizadas: {len(df)}")
print(f"Número de WSIs únicas: {df['archivo'].nunique()}")
print(f"Folds detectados: {sorted(df['fold'].unique())}\n")

# Agrupamos por archivo y contamos cuántos folds únicos tiene asignados cada uno
wsi_fold_counts = df.groupby("archivo")["fold"].nunique()
leaked_wsis = wsi_fold_counts[wsi_fold_counts > 1]

if len(leaked_wsis) == 0:
    print("Ninguna WSI está repetida en diferentes folds.")
else:
    print("¡DATA LEAKAGE!")
    print(f"Se encontraron {len(leaked_wsis)} WSIs compartidas entre distintos folds:")
    print(leaked_wsis)

Total de filas analizadas: 200
Número de WSIs únicas: 200
Folds detectados: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Ninguna WSI está repetida en diferentes folds.


In [13]:
# =========================
# ESTADÍSTICAS FOLDS
# =========================

df_analysis = pd.read_csv(STATISTICS_FOLDER / "analysis_wsi_folds.csv")

# Dataframe a diccionario
map_folds = dict(zip(df_analysis["archivo"], df_analysis["fold"]))

# Estadisticas de patches por clase por fold
statistics_folds = np.zeros((5, 7), dtype=int)

# Numero patches por fold
patches_per_fold = np.zeros(5, dtype=int)

for h5_path in H5_FILES:
    name_path = os.path.basename(h5_path)
    
    # Si archivo no está en el mapa, saltarlo
    if name_path not in map_folds:
        print(name_path)
        continue
        
    fold_assigned = map_folds[name_path]
    
    with h5py.File(h5_path, "r") as f:
        labels = f["labels"][:] 
        
    # Sumar las columnas para saber cuántos parches hay de cada clase en este archivo
    classes_per_wsi = np.sum(labels, axis=0)
    
    # Acumular en Fold correspondiente
    statistics_folds[fold_assigned] += classes_per_wsi.astype(int)
    patches_per_fold[fold_assigned] += len(labels)

df_resultados = pd.DataFrame(statistics_folds, columns=CLASS_NAMES)
df_resultados.insert(0, "Total Patches", patches_per_fold)
df_resultados.index.name = "Fold ID"

# Guardar CSV
df_resultados.to_csv(STATISTICS_FOLDER / "statistics_per_fold.csv")

print(df_resultados.to_string())

         Total Patches  normal  lowgrade_dysplasia  inflammation  highgrade_dysplasia  tumor_necrosis  suspicious_for_invasion  adenocarcinoma
Fold ID                                                                                                                                       
0                19502    7366               10494           264                  850              89                       70             627
1                19848    4162               12781           332                  581             186                       73            2035
2                20480    5963               12974           130                  537              81                      303             544
3                19284    7223               10900           220                  587             170                      107             691
4                16644    6245                8999            80                  492              96                      120             664

In [14]:
# =========================
# ESTADÍSTICAS FOLDS MULTILABEL
# =========================


df_analysis = pd.read_csv(STATISTICS_FOLDER / "analysis_wsi_folds.csv")

# Dataframe a diccionario
map_folds = dict(zip(df_analysis["archivo"], df_analysis["fold"]))

# Diccionario para almacenar los conteos de cada combinación por fold
count_per_fold = {0: Counter(), 1: Counter(), 2: Counter(), 3: Counter(), 4: Counter()}

for h5_path in H5_FILES:
    name_path = os.path.basename(h5_path)
    
    if name_path not in map_folds:
        continue
        
    fold_assigned = map_folds[name_path]
    
    with h5py.File(h5_path, "r") as f:
        labels = f["labels"][:]
        
        # Procesar cada patch de la WSI actual
        for fila in labels:
            if np.sum(fila) > 1:
                # Obtener los nombres de las clases donde hay 1
                active_classes = [CLASS_NAMES[i] for i, valor in enumerate(fila) if valor == 1]
                
                name_combination = " + ".join(active_classes)
                
                # Acumular en el contador del fold correspondiente
                count_per_fold[fold_assigned][name_combination] += 1

# Lista única de todas las combinaciones multilabel encontradas en el dataset
all_combinations = set()
for c in count_per_fold.values():
    all_combinations.update(c.keys())
    
all_combinations = sorted(list(all_combinations))

# Crear la matriz de datos
data_matrix = np.zeros((len(all_combinations), 5), dtype=np.int32)

for col_fold in range(5):
    for row_idx, comb in enumerate(all_combinations):
        data_matrix[row_idx, col_fold] = count_per_fold[col_fold][comb]


df_final_combinations = pd.DataFrame(
    data_matrix, 
    index=all_combinations, 
    columns=[f"Fold {i}" for i in range(5)]
)

# Añadir una columna con el total absoluto para facilitar la lectura
df_final_combinations["Total Global"] = df_final_combinations.sum(axis=1)
df_final_combinations = df_final_combinations.sort_values(by="Total Global", ascending=False)

# Guardar CSV
df_final_combinations.to_csv(STATISTICS_FOLDER / "multilabel_per_fold.csv")

print(df_final_combinations.to_string())

                                                Fold 0  Fold 1  Fold 2  Fold 3  Fold 4  Total Global
lowgrade_dysplasia + highgrade_dysplasia            68     203      20     450       0           741
tumor_necrosis + adenocarcinoma                      8      61       0      98       9           176
lowgrade_dysplasia + inflammation                  153       3       0       1       6           163
tumor_necrosis + suspicious_for_invasion            23      13      22      18       0            76
inflammation + adenocarcinoma                        0       9       9      38       2            58
inflammation + highgrade_dysplasia                   1       1       0       1      18            21
inflammation + suspicious_for_invasion               4      12       0       0       0            16
lowgrade_dysplasia + suspicious_for_invasion         0       0       0       0      10            10
highgrade_dysplasia + adenocarcinoma                 0       0       1       0       3     